# CityTrack S02 Ablation R4 - all 3

Slug: `mrkdagods/citytrack-s02-all`

Mode: GPU, self-contained stages 0-5

Commit pin: `paper-tests@b5aef3e`

S02 camera scope: `S02_c006`, `S02_c007`, `S02_c008`

Overrides: `stage1.ssa.enabled=true stage1.bidirectional.enabled=true stage4.association.occlusion_aware.enabled=true`

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import time
import tarfile
from pathlib import Path

RUN_LABEL = "R4"
KERNEL_SLUG = "citytrack-s02-all"
RUN_ID = "citytrack_s02_all"
RUN_OVERRIDES = ["stage1.ssa.enabled=true", "stage1.bidirectional.enabled=true", "stage4.association.occlusion_aware.enabled=true"]
TARGET_CAMERAS = ["S02_c006", "S02_c007", "S02_c008"]
REPO_URL = "https://github.com/MRKDaGods/gp.git"
BRANCH = "paper-tests"
EXPECTED_COMMIT = "b5aef3e"
WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"
DATA_OUT = Path("/tmp/pipeline_outputs")

if shutil.which("nvidia-smi"):
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.returncode == 0 and result.stdout.strip():
        gpu_name, compute_cap = result.stdout.strip().split(",", 1)
        match = re.search(r"(\d+)\.(\d+)", compute_cap)
        if match:
            major, minor = match.groups()
            sm = int(major) * 10 + int(minor)
            if sm < 70:
                print(f"GPU {gpu_name.strip()} sm_{sm}: installing torch 2.4.1+cu124")
                subprocess.check_call([
                    sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.4.1+cu124", "torchvision==0.19.1+cu124",
                    "--index-url", "https://download.pytorch.org/whl/cu124",
                ])

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f"  GPU {idx}: {torch.cuda.get_device_name(idx)} ({props.total_memory / 1024**3:.1f} GB)")
if not torch.cuda.is_available():
    raise RuntimeError("This ablation run requires a Kaggle GPU kernel")

if PROJECT.exists():
    shutil.rmtree(PROJECT)
subprocess.check_call(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJECT)])
try:
    subprocess.check_call(["git", "-C", str(PROJECT), "checkout", EXPECTED_COMMIT])
except subprocess.CalledProcessError:
    subprocess.check_call(["git", "-C", str(PROJECT), "fetch", "origin", EXPECTED_COMMIT, "--depth", "1"])
    subprocess.check_call(["git", "-C", str(PROJECT), "checkout", EXPECTED_COMMIT])
os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
head_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("git rev-parse HEAD:", head_sha)
if not head_sha.startswith(EXPECTED_COMMIT):
    raise RuntimeError(f"Expected {EXPECTED_COMMIT}, got {head_sha}")
print(f"Repo ready at {PROJECT}")

## Install Dependencies

In [ ]:
def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


# Keep Kaggle's torch stack unless the setup cell pinned it for P100.
pip_install("--no-deps", "ultralytics")
pip_install("filterpy", "ftfy", "lapx")
pip_install("--no-deps", "boxmot==11.0.3")

try:
    import torchreid
    print("torchreid ok")
except ImportError:
    pip_install("git+https://github.com/KaiyangZhou/deep-person-reid.git")

try:
    import faiss
    print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try:
        pip_install("faiss-gpu")
    except Exception:
        pip_install("faiss-cpu")

try:
    import trackeval
    print("trackeval ok")
except ImportError:
    pip_install("git+https://github.com/JonathonLuiten/TrackEval.git")

pip_install("timm", "motmetrics")
pip_install("loguru", "omegaconf", "rich", "networkx>=3.1", "click", "scipy", "pandas", "scikit-learn")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=str(PROJECT))

FAILED = []
for label, module in [
    ("ultralytics", "ultralytics"),
    ("boxmot", "boxmot"),
    ("torch", "torch"),
    ("torchreid", "torchreid"),
    ("timm", "timm"),
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("trackeval", "trackeval"),
    ("cv2", "cv2"),
    ("omegaconf", "omegaconf"),
    ("networkx", "networkx"),
    ("sklearn", "sklearn"),
]:
    try:
        __import__(module)
        print(f"  OK {label}")
    except ImportError as exc:
        print(f"  MISSING {label}: {exc}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED}")
print("All dependencies importable")

## Prepare Weights

In [ ]:
import zipfile


def first_existing(paths):
    return next((path for path in paths if path.exists()), None)


weights_candidates = [
    Path("/kaggle/input/mtmc-weights"),
    Path("/kaggle/input/mrkdagods-mtmc-weights"),
    Path("/kaggle/input/mrkdagods/mtmc-weights"),
    Path("/kaggle/input/gumfreddy-mtmc-weights"),
    Path("/kaggle/input/gumfreddy/mtmc-weights"),
    Path("/kaggle/input/yahiaakhalafallah-mtmc-weights"),
    Path("/kaggle/input/yahiaakhalafallah/mtmc-weights"),
]
WEIGHTS_INPUT = first_existing(weights_candidates)
if WEIGHTS_INPUT is None and Path("/kaggle/input").exists():
    candidates = [
        path for path in Path("/kaggle/input").rglob("*")
        if path.is_dir() and "mtmc" in path.name.lower() and "weight" in path.name.lower()
    ]
    WEIGHTS_INPUT = candidates[0] if candidates else None
if WEIGHTS_INPUT is None:
    raise FileNotFoundError("MTMC weights dataset not found in /kaggle/input")
print(f"MTMC weights input: {WEIGHTS_INPUT}")

MODELS_DST = PROJECT / "models"
if MODELS_DST.is_symlink():
    MODELS_DST.unlink()
if MODELS_DST.exists():
    shutil.rmtree(MODELS_DST)
shutil.copytree(str(WEIGHTS_INPUT), str(MODELS_DST))

nested_models = MODELS_DST / "models"
if nested_models.exists() and nested_models.is_dir():
    for child in nested_models.iterdir():
        target = MODELS_DST / child.name
        if target.exists():
            if target.is_dir():
                shutil.rmtree(target)
            else:
                target.unlink()
        shutil.move(str(child), str(target))
    shutil.rmtree(nested_models)

for zip_path in sorted(MODELS_DST.rglob("*.zip")):
    print(f"Extracting {zip_path.relative_to(MODELS_DST)}")
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(zip_path.parent)
    zip_path.unlink()

for subdir in ["detection", "reid", "tracker"]:
    (MODELS_DST / subdir).mkdir(exist_ok=True)
for path in list(MODELS_DST.glob("*.pt")):
    if "yolo" in path.name.lower():
        shutil.move(str(path), str(MODELS_DST / "detection" / path.name))
    elif "osnet" in path.name.lower():
        shutil.move(str(path), str(MODELS_DST / "tracker" / path.name))
for path in list(MODELS_DST.glob("*.pth")):
    shutil.move(str(path), str(MODELS_DST / "reid" / path.name))
for path in list(MODELS_DST.glob("*.pkl")):
    shutil.move(str(path), str(MODELS_DST / "reid" / path.name))
for path in list(MODELS_DST.glob("*.json")):
    if path.name != "dataset-metadata.json":
        shutil.move(str(path), str(MODELS_DST / "reid" / path.name))

required = [
    PROJECT / "models" / "detection" / "yolo26m.pt",
    PROJECT / "models" / "reid" / "transreid_cityflowv2_best.pth",
    PROJECT / "models" / "tracker" / "osnet_x0_25_msmt17.pt",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required weights: {missing}")

dinov2_name = "vehicle_transreid_dinov2_large_cityflowv2_final.pth"
dinov2_candidates = [
    Path("/kaggle/input/dinov2-large-cityflowv2-mrk") / dinov2_name,
    Path("/kaggle/input/mrkdagods-dinov2-large-cityflowv2-mrk") / dinov2_name,
    Path("/kaggle/input/mrkdagods/dinov2-large-cityflowv2-mrk") / dinov2_name,
]
TERTIARY_WEIGHTS = first_existing(dinov2_candidates)
if TERTIARY_WEIGHTS is None and Path("/kaggle/input").exists():
    matches = list(Path("/kaggle/input").rglob(dinov2_name))
    TERTIARY_WEIGHTS = matches[0] if matches else None
if TERTIARY_WEIGHTS is None:
    raise FileNotFoundError(f"DINOv2 tertiary checkpoint not found: {dinov2_name}")
print(f"DINOv2 tertiary weights: {TERTIARY_WEIGHTS}")

## Scope CityFlowV2 To S02

In [ ]:
for mount in ["/tmp", "/kaggle/working"]:
    total, used, free = shutil.disk_usage(mount)
    print(f"{mount:16s} {free / 1024**3:.1f} GB free / {total / 1024**3:.1f} GB total")

candidate_mounts = [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]
CITYFLOW_INPUT = next((path for path in candidate_mounts if path.exists()), None)
if CITYFLOW_INPUT is None:
    raise FileNotFoundError("CityFlowV2 dataset not found; attach thanhnguyenle/data-aicity-2023-track-2")
print(f"CityFlowV2 input: {CITYFLOW_INPUT}")

TMP_DATA = Path("/tmp/datasets")
TMP_DATA.mkdir(parents=True, exist_ok=True)
DATA_RAW_PARENT = PROJECT / "data" / "raw"
if DATA_RAW_PARENT.exists() or DATA_RAW_PARENT.is_symlink():
    if DATA_RAW_PARENT.is_symlink() or DATA_RAW_PARENT.is_file():
        DATA_RAW_PARENT.unlink()
    else:
        shutil.rmtree(DATA_RAW_PARENT)
DATA_RAW_PARENT.parent.mkdir(parents=True, exist_ok=True)
DATA_RAW_PARENT.symlink_to(TMP_DATA)

DATA_RAW = TMP_DATA / "cityflowv2"
if DATA_RAW.exists():
    shutil.rmtree(DATA_RAW)
DATA_RAW.mkdir(parents=True, exist_ok=True)

for split_dir in sorted(CITYFLOW_INPUT.iterdir()):
    if not split_dir.is_dir() or split_dir.name not in ("train", "validation", "test"):
        continue
    for scene_dir in sorted(split_dir.iterdir()):
        if not scene_dir.is_dir() or scene_dir.name != "S02":
            continue
        for cam_dir in sorted(scene_dir.iterdir()):
            if not cam_dir.is_dir():
                continue
            flat_name = f"{scene_dir.name}_{cam_dir.name}"
            if flat_name not in TARGET_CAMERAS:
                continue
            flat_dir = DATA_RAW / flat_name
            if not flat_dir.exists():
                flat_dir.symlink_to(cam_dir)

present = sorted(path.name for path in DATA_RAW.iterdir() if path.is_dir())
print(f"S02 camera scope: {present}")
missing_cams = sorted(set(TARGET_CAMERAS) - set(present))
if missing_cams:
    raise FileNotFoundError(f"Missing S02 cameras in flattened dataset: {missing_cams}")

print("Generating ROI masks for S02 cameras")
roi_result = subprocess.run(
    [sys.executable, "scripts/generate_roi_masks.py", "--data-dir", str(DATA_RAW), "--n-samples", "200"],
    cwd=str(PROJECT),
)
if roi_result.returncode != 0:
    print("WARNING: ROI mask generation failed; continuing with existing/no masks")
else:
    print(f"ROI masks ready: {len(list(DATA_RAW.glob('*/roi.jpg')))}")

## Run Stages 0-5

In [ ]:
os.chdir(str(PROJECT))
RUN_DIR = DATA_OUT / RUN_ID
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
DATA_OUT.mkdir(parents=True, exist_ok=True)

base_overrides = [
    f"project.run_name={RUN_ID}",
    f"project.output_dir={DATA_OUT}",
    "stage0.cameras=[S02_c006,S02_c007,S02_c008]",
    f"stage5.ground_truth_dir={DATA_RAW}",
    f"stage2.reid.vehicle3.weights_path={TERTIARY_WEIGHTS}",
    f"stage4.association.tertiary_embeddings.path={RUN_DIR / 'stage2' / 'embeddings_tertiary.npy'}",
]
all_overrides = base_overrides + RUN_OVERRIDES

cmd = [
    sys.executable,
    "scripts/run_pipeline.py",
    "--config",
    "configs/default.yaml",
    "--dataset-config",
    "configs/datasets/cityflowv2.yaml",
    "--stages",
    "0,1,2,3,4,5",
]
for item in all_overrides:
    cmd += ["--override", item]

manifest = {
    "run_label": RUN_LABEL,
    "kernel_slug": KERNEL_SLUG,
    "run_id": RUN_ID,
    "commit": EXPECTED_COMMIT,
    "target_cameras": TARGET_CAMERAS,
    "overrides": RUN_OVERRIDES,
    "all_overrides": all_overrides,
    "stages": "0,1,2,3,4,5",
    "mode": "single_kernel_gpu",
}
(WORK_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2))

print("CMD:", " ".join(str(part) for part in cmd))
print("=" * 80)
start = time.time()
result = subprocess.run(cmd, cwd=str(PROJECT))
elapsed_min = (time.time() - start) / 60.0
print("=" * 80)
if result.returncode != 0:
    raise SystemExit(result.returncode)
print(f"Pipeline completed in {elapsed_min:.1f} min")

## Stage-5 Results

In [ ]:
stage5_dir = RUN_DIR / "stage5"
metrics_path = stage5_dir / "evaluation_report.json"
if not metrics_path.exists():
    raise FileNotFoundError(metrics_path)

metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
details = metrics.get("details", {}) or {}
per_camera = details.get("per_camera", {}) or {}
mtmc_idf1 = metrics.get("mtmc_idf1") or details.get("mtmc_idf1") or metrics.get("idf1")

summary = {
    "run_label": RUN_LABEL,
    "kernel_slug": KERNEL_SLUG,
    "run_id": RUN_ID,
    "commit": EXPECTED_COMMIT,
    "target_cameras": TARGET_CAMERAS,
    "overrides": RUN_OVERRIDES,
    "mtmc_idf1": mtmc_idf1,
    "idf1": metrics.get("idf1"),
    "mota": metrics.get("mota"),
    "hota": metrics.get("hota"),
    "id_switches": metrics.get("id_switches"),
    "details_mtmc_idf1": details.get("mtmc_idf1"),
    "details_mtmc_mota": details.get("mtmc_mota"),
    "details_mtmc_id_switches": details.get("mtmc_id_switches"),
    "per_camera": per_camera,
    "s02_c006": per_camera.get("S02_c006"),
    "metrics_path": str(metrics_path),
}

print("S02 MTMC IDF1:", mtmc_idf1)
print("Per-camera metrics:")
for camera_id in TARGET_CAMERAS:
    print(f"  {camera_id}: {per_camera.get(camera_id)}")

summary_path = WORK_DIR / f"{KERNEL_SLUG}_results.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
shutil.copy2(metrics_path, WORK_DIR / f"{KERNEL_SLUG}_evaluation_report.json")
html_report = stage5_dir / "evaluation_report.html"
if html_report.exists():
    shutil.copy2(html_report, WORK_DIR / f"{KERNEL_SLUG}_evaluation_report.html")

stage5_tar = WORK_DIR / f"{KERNEL_SLUG}_stage5.tar.gz"
with tarfile.open(str(stage5_tar), "w:gz") as tar:
    tar.add(str(stage5_dir), arcname="stage5")
print(f"Wrote {summary_path}")
print(f"Packed Stage-5 outputs: {stage5_tar}")